<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Reproducible extraction

**[Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/)** · SQL and Data Preparation for AI · Module 3, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** run saved SQL queries from Python with named parameters, write the results to CSV files, and record the snapshot, the selection, a SHA-256 of each query and the row counts in a manifest, so anybody can repeat the extraction.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | Half-open date ranges and "one latest row per ticket", from [Time and ranking](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/). From the Python course: read a text file with `pathlib` and write a CSV file with the `csv` module, from [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#paths-with-pathlib) (sections *Paths with pathlib* and *CSV and JSON*). |
| **You do not need** | The local project from Module 1. The setup below builds the same database in a folder `ticket-data`, and you write the query files and the script there, as in the lesson. |
| **Tested** | Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, file, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/) has the full explanations, the line-by-line tables for the script and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M03-L03-reproducible-extraction/reproducible-extraction-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A file that nobody can explain

Priya, the data scientist, receives a CSV file of 1,416 tickets. Which export is it from? Which tickets were selected, and why not 1,417? Which query made it, and when? The file cannot answer.

So Mei, the data engineer, never sends a data file alone. She sends a **manifest** with it: a small JSON file that records the extraction time, the source version (the `snapshot` table), the selection criteria, a SHA-256 of each query, and the row counts. A **SHA-256** is a 64-character fingerprint of a text. The same text always gives the same fingerprint; a change of one character gives a completely different one.

In this notebook you build Mei's extraction in the folder `ticket-data`, as on your computer: two query files in `queries/`, a script `scripts/extract.py`, and its output in `data/extract/`. The [lesson page](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/#a-file-that-nobody-can-explain) has the table of the five facts.

## 2. Parameters: values that are not part of the query

The selection dates change from one extraction to the next. The query should not. So Mei writes the query with **named parameters**: `:team` is a placeholder, and Python gives its value separately, in a dictionary.

The setup cell opened the database as `con`, **read-only**, as `q.py` does: a query in this notebook cannot change the data.

> **Predict.** How many tickets does the returns team have? Then run the cell.

In [ ]:
query = "SELECT COUNT(*) FROM tickets WHERE team = :team"
con.execute(query, {"team": "returns"}).fetchone()[0]

> **Check.** You should see **327**.

> **Your turn: change one thing.** In the cell above, change `"returns"` to `"payment"`, and run it again: 241. The query text did not change, only the value.

`sqlite3` sends the query text and the values to SQLite separately. A value can never become part of the SQL. This also keeps the query text the same for every selection, so its SHA-256 stays the same. **Never put a value into SQL text with an f-string.** Section 7 shows why.

## 3. Preview the selection in SQL

Before Mei writes the script, she checks what the selection will contain. She puts the two dates in place of the parameters, only for this test. A comparison is 1 when true and 0 when false, so `SUM` counts the true rows.

> **Predict.** How many ticket rows are not selected? Then run the cell.

In [ ]:
sql("""
SELECT
    SUM(created_at >= '2026-01-01' AND created_at < '2026-07-01') AS selected,
    SUM(NOT (created_at >= '2026-01-01' AND created_at < '2026-07-01')) AS not_selected,
    COUNT(*) AS all_rows
FROM tickets;
""")

> **Check.** You should see `1416`, `1`, `1417`. Run the next cell to see the row that is not selected.

In [ ]:
sql("SELECT ticket_id, created_at FROM tickets WHERE created_at >= '2026-07-01';")

> **Check.** You should see T-10147, the ticket from 2036.

The extraction keeps the 14 repeated ticket rows. Removing them is a cleaning decision, and Module 4 makes it and counts it. An extraction only selects: it does not change values.

## 4. The two query files

Mei saves each query in its own file, so that the script can compute its SHA-256 and a reviewer can read it. First, make the two folders of the project. On your computer, `queries/` is already there, and you make `scripts/` with `mkdir scripts`. Run the cell.

In [ ]:
Path("queries").mkdir(exist_ok=True)
Path("scripts").mkdir(exist_ok=True)
print("Working folder:", Path.cwd().name)

> **Check.** You should see `Working folder: ticket-data`.

A cell that starts with `%%writefile <file>` saves the rest of the cell as that file, and prints `Writing …` (or `Overwriting …` the second time). Run the next two cells. Do not change the text: the SHA-256 depends on every character.

In [ ]:
%%writefile queries/extract_tickets.sql
-- One row per ticket row in the export, with the customer's attributes.
-- LEFT JOIN keeps tickets with no customer (guests) and tickets whose
-- customer is no longer in the customer table. Duplicates are kept here:
-- the cleaning step removes them and counts them.
SELECT
    t.ticket_id,
    t.customer_id,
    t.created_at,
    t.channel,
    t.team,
    t.priority,
    t.order_value,
    t.word_count,
    t.first_reply_minutes,
    t.priority_now,
    t.closed_at,
    c.segment,
    c.region,
    c.joined_on
FROM tickets AS t
LEFT JOIN customers AS c
    ON c.customer_id = t.customer_id
WHERE t.created_at >= :created_from
  AND t.created_at <  :created_before
ORDER BY t.ticket_id;

In [ ]:
%%writefile queries/extract_outcomes.sql
-- Every outcome event of the tickets in the selection. Events for tickets
-- that are not in the ticket table are left out by the join.
SELECT
    o.outcome_id,
    o.ticket_id,
    o.recorded_at,
    o.status,
    o.resolution_code,
    o.csat
FROM outcomes AS o
WHERE o.ticket_id IN (
    SELECT t.ticket_id
    FROM tickets AS t
    WHERE t.created_at >= :created_from
      AND t.created_at <  :created_before
)
ORDER BY o.outcome_id;

Three choices in these files matter:

- **The same rule in both files.** An outcome event is selected when its ticket is selected, so the two files always describe the same tickets.
- **`ORDER BY`.** Without it, the database can return rows in any order, and the CSV file can differ between two runs with the same data.
- **The extraction keeps leakage columns such as `priority_now`.** The extraction is a faithful copy; Module 5 decides which columns a model may use.

The outcomes query uses `IN (subquery)`, not a join. `IN` only tests if the ticket is in the list, so a ticket that is in `tickets` twice cannot double its events.

Run the cell: it computes the SHA-256 of each file, as the script will, and compares it with the lesson's value.

In [ ]:
import hashlib
for file, expected in [("queries/extract_tickets.sql", "28a0267e280062b0e02f9727cb5a95d2c8b60309e5fc992a578e44ba61b54338"),
                       ("queries/extract_outcomes.sql", "533e9573c06975a463369aea503c4218c806640b99067d240607c1cea6fdc864")]:
    sha = hashlib.sha256(Path(file).read_text(encoding="utf-8").encode()).hexdigest()
    expect(f"the SHA-256 of {file}", sha, expected)

> **Check.** You should see `Check passed` twice. If you changed one character in a file, run its `%%writefile` cell again.

> **Your turn.** How many outcome events will the extraction keep? The query counts all 1,874 events. Add the selection rule of `extract_outcomes.sql`, with the two dates in place of the parameters. Predict first: 4 events belong to tickets that are not in `tickets`, and the 2036 ticket has 1 event. Then run the check cell.

In [ ]:
selected_events = """
-- Count the outcome events whose ticket is in the selection
-- (created from 2026-01-01, before 2026-07-01).
SELECT COUNT(*) AS events
FROM outcomes;
"""
sql(selected_events)

In [ ]:
check(selected_events, rows=1, columns=1, expected="c5d8505c1aecffdf")

## 5. Guided practice: build extract.py

Build the script in three versions. Run each version before you add more.

**Version 1** runs one query with parameters and prints the row count. The address `file:larkfield.db?mode=ro` with `uri=True` opens the database read-only. The keys of `params` are the parameter names without the colon. Run the cell to save the script.

In [ ]:
%%writefile scripts/extract.py
"""Extract tickets and outcomes from larkfield.db into CSV files, with a manifest."""

import sqlite3
from pathlib import Path


def main():
    params = {"created_from": "2026-01-01", "created_before": "2026-07-01"}
    sql = Path("queries/extract_tickets.sql").read_text(encoding="utf-8")

    con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
    rows = con.execute(sql, params).fetchall()
    con.close()

    print(f"{len(rows)} ticket rows")
    print(rows[0])


if __name__ == "__main__":
    main()

Now run the script. A line that starts with `!` runs a command in a terminal, here in the folder `ticket-data`. `{sys.executable}` is the Python of this notebook. On your computer, you type `python3 scripts/extract.py` in the project folder.

> **Predict.** How many ticket rows? Then run the cell.

In [ ]:
!"{sys.executable}" scripts/extract.py

> **Check.** You should see `1416 ticket rows`, then the first row, which starts `('T-10001', 'C-0228', '2026-01-05 07:30:24', 'email', 'returns', …`.

If you see `sqlite3.ProgrammingError: You did not supply a value for binding parameter :created_before.`, a key in `params` is missing or has a different spelling.

**Version 2** adds the function `run`. It runs one query file, writes a CSV file, and returns the facts for the manifest: the query file, its SHA-256, the output file and the row count. `cursor.description` gives the column names for the CSV header. `lineterminator="\n"` gives the same line ends on every computer. Run the cell to replace the script, then run the next cell.

In [ ]:
%%writefile scripts/extract.py
"""Extract tickets and outcomes from larkfield.db into CSV files, with a manifest."""

import csv
import hashlib
import sqlite3
from pathlib import Path

OUT = Path("data/extract")


def run(con, query_file, params, out_file):
    sql = Path(query_file).read_text(encoding="utf-8")
    cursor = con.execute(sql, params)
    header = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    with open(out_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(header)
        writer.writerows(rows)
    return {"query": query_file, "query_sha256": hashlib.sha256(sql.encode()).hexdigest(),
            "file": str(out_file), "rows": len(rows)}


def main():
    params = {"created_from": "2026-01-01", "created_before": "2026-07-01"}
    con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
    OUT.mkdir(parents=True, exist_ok=True)
    files = [
        run(con, "queries/extract_tickets.sql", params, OUT / "tickets.csv"),
        run(con, "queries/extract_outcomes.sql", params, OUT / "outcomes.csv"),
    ]
    con.close()
    for f in files:
        print(f)


if __name__ == "__main__":
    main()

In [ ]:
!"{sys.executable}" scripts/extract.py

> **Check.** You should see two dictionaries: `queries/extract_tickets.sql` with `'query_sha256': '28a0267e…'` and `'rows': 1416`, and `queries/extract_outcomes.sql` with `'533e9573…'` and `'rows': 1869`.

**The final version** adds three things: the dates as command-line options (`argparse`), the snapshot facts from the database, and the manifest. The [lesson page](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/#guided-practice-build-extractpy) explains each part in a table. Run the cell to replace the script, then run the next cell.

In [ ]:
%%writefile scripts/extract.py
"""Extract tickets and outcomes from larkfield.db into CSV files, with a manifest."""

import argparse
import csv
import hashlib
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

OUT = Path("data/extract")


def run(con, query_file, params, out_file):
    sql = Path(query_file).read_text(encoding="utf-8")
    cursor = con.execute(sql, params)
    header = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    with open(out_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(header)
        writer.writerows(rows)
    return {"query": query_file, "query_sha256": hashlib.sha256(sql.encode()).hexdigest(),
            "file": str(out_file), "rows": len(rows)}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--created-from", default="2026-01-01")
    parser.add_argument("--created-before", default="2026-07-01")
    args = parser.parse_args()
    params = {"created_from": args.created_from, "created_before": args.created_before}

    con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
    snapshot = dict(con.execute("SELECT name, value FROM snapshot").fetchall())
    total = con.execute("SELECT COUNT(*) FROM tickets").fetchone()[0]
    OUT.mkdir(parents=True, exist_ok=True)
    files = [
        run(con, "queries/extract_tickets.sql", params, OUT / "tickets.csv"),
        run(con, "queries/extract_outcomes.sql", params, OUT / "outcomes.csv"),
    ]
    con.close()

    manifest = {
        "extracted_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source": snapshot,
        "selection": {**params, "rule": "created_from <= created_at < created_before"},
        "ticket_rows_in_source": total,
        "ticket_rows_not_selected": total - files[0]["rows"],
        "files": files,
    }
    (OUT / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    for f in files:
        print(f"{f['file']}: {f['rows']} rows")
    print(f"Not selected: {manifest['ticket_rows_not_selected']} ticket row(s)")


if __name__ == "__main__":
    main()

In [ ]:
!"{sys.executable}" scripts/extract.py

> **Check.** You should see `data/extract/tickets.csv: 1416 rows`, `data/extract/outcomes.csv: 1869 rows` and `Not selected: 1 ticket row(s)`.

Now read the manifest. Run the cell.

In [ ]:
print(Path("data/extract/manifest.json").read_text(encoding="utf-8"))

> **Check.** Your `extracted_at` is your own time, in UTC. Everything else is the same as on the lesson page: the source `exported_at` is `2026-07-01 06:00:00` and `data_version` is `1.0`, the selection has the two dates and the rule, `ticket_rows_in_source` is 1417, `ticket_rows_not_selected` is 1, and the two files have the SHA-256 values and row counts of version 2.

The manifest answers Priya's four questions. Empty cells in the CSV files are NULL values in the database. Module 4 loads them with care.

## 6. Your turn: extract March only

Change one condition: extract the tickets of March 2026 only. First, run the cell: it keeps the full manifest, to compare it later.

In [ ]:
import json
full = json.loads(Path("data/extract/manifest.json").read_text(encoding="utf-8"))
print("Kept the manifest of", full["selection"]["created_from"], "to", full["selection"]["created_before"])

> **Your turn.** Add the two options `--created-from 2026-03-01 --created-before 2026-04-01` to the command in the next cell, and run it. Predict first: how many ticket rows? You counted March in the last lesson. Then run the check cell.

In [ ]:
!"{sys.executable}" scripts/extract.py  # add the two options here

In [ ]:
march = json.loads(Path("data/extract/manifest.json").read_text(encoding="utf-8"))
expect("created_from in the manifest", march["selection"]["created_from"], "2026-03-01")
expect("created_before in the manifest", march["selection"]["created_before"], "2026-04-01")
same = [f["query_sha256"] for f in march["files"]] == [f["query_sha256"] for f in full["files"]]
print("The query SHA-256 values are the same as in the full extraction:", same)

The script shows the March row counts and the rows not selected. The two `query_sha256` values do **not** change, because the dates are parameters and not part of the query text. The March files replace the earlier files in `data/extract/`. Without the manifest, nobody could tell which selection the folder holds now.

Run the script again with no options, to restore the full extraction.

In [ ]:
!"{sys.executable}" scripts/extract.py

> **Check.** You should see 1416 rows, 1869 rows and 1 row not selected again.

## 7. Failure case: an f-string query breaks on a quote, or returns every row

Tomás builds a query for one team with an f-string. It works for weeks. Then it stops with an error for one value, and for another value it returns all 1,417 rows with no error. On your computer, the lesson puts this code in a scratch file `scripts/try_quote.py`. Here, run it in the cell.

> **Predict.** For each of the three values, what do the parameter query and the f-string query return? Then run the cell.

In [ ]:
for team in ["returns", "customer's account", "returns' OR '1'='1"]:
    print("value:", team)
    safe = con.execute("SELECT COUNT(*) FROM tickets WHERE team = :team", {"team": team}).fetchone()[0]
    print("  parameter:", safe)
    try:
        unsafe = con.execute(f"SELECT COUNT(*) FROM tickets WHERE team = '{team}'").fetchone()[0]
        print("  f-string: ", unsafe)
    except sqlite3.Error as error:
        print("  f-string:  error:", type(error).__name__, error)

> **Check.** You should see 327 and 327 for `returns`. For `customer's account`: 0 with the parameter, and `OperationalError near "s": syntax error` with the f-string. For the third value: 0 with the parameter, and **1417** with the f-string.

**Cause:** the f-string puts the value inside the SQL text. The quote in `customer's` ends the SQL string early, so SQLite reads `s account'` as SQL, and fails. The third value turns into SQL code, `team = 'returns' OR '1'='1'`, which is true for every row. Attackers use this method, **SQL injection**, to read or change data that they must not reach. (Here `con` is read-only, so an injected `DELETE` would fail. Do not depend on that.)

**Fix:** use a parameter for every value. With a parameter, both strange values are only text: no team has that name, so the count is 0.

> **Warning.** In AI work, values often come from outside: a date from a scheduler, a customer name, a ticket text, the output of a language model. Any of them can contain a quote. A parameter keeps each one a value.

## 8. Module practice: one latest outcome per ticket

The task of this module: **extract one latest outcome per ticket, without accidental duplication.** There are two quiet ways to get more rows than tickets: ties (the failure case of the last lesson), and a join to the repeated ticket rows. Mei joins the latest events (1,399 rows) to `tickets`.

> **Predict.** Are the result rows and the distinct tickets equal? Then run the cell.

In [ ]:
sql("""
WITH ranked AS (
    SELECT ticket_id, outcome_id, recorded_at, status,
           ROW_NUMBER() OVER (
               PARTITION BY ticket_id
               ORDER BY recorded_at DESC, outcome_id DESC
           ) AS rn
    FROM outcomes
),
latest AS (
    SELECT * FROM ranked WHERE rn = 1
)
SELECT COUNT(*) AS result_rows, COUNT(DISTINCT t.ticket_id) AS tickets
FROM tickets AS t
JOIN latest AS l
    ON l.ticket_id = t.ticket_id;
""")

> **Check.** You should see `1409` rows for `1395` tickets. The 14 tickets that appear twice in `tickets` each match their latest event twice. The join also removed the 4 events of tickets that are not in `tickets` (1,399 → 1,395).

The fix is to join to a list that has one row per ticket: `SELECT DISTINCT ticket_id FROM tickets`. Then the check is: **result rows = distinct tickets = tickets that have events = 1,395** (1,403 distinct tickets, minus the 8 with no event).

> **Your turn.** Write a query that returns one row per ticket in the export that has at least one outcome event. Return four columns: `ticket_id`, `outcome_id`, `recorded_at` and `status` of the latest event. Break ties with `outcome_id`. The result must have 1,395 rows. The check compares every row, so a wrong tie choice (for example, `reopened` for T-10136) fails even when the count is right.

On your computer, save it as `queries/35-latest-outcome.sql`.

In [ ]:
latest_outcome = """
-- One latest outcome event per ticket in the export.
-- Columns: ticket_id, outcome_id, recorded_at, status.
-- Write your query here.
SELECT ticket_id, outcome_id, recorded_at, status
FROM outcomes
LIMIT 5;
"""
sql(latest_outcome)

In [ ]:
check(latest_outcome, rows=1395, columns=4, expected="dc0e038df0a255bf")

Now check the grain. Run the cell: it counts the rows, the distinct tickets and the statuses of your result.

In [ ]:
from collections import Counter
columns, rows = run(latest_outcome)
print("rows:", len(rows), " distinct tickets:", len({row[0] for row in rows}))
print(Counter(row[3] for row in rows if len(row) > 3))

> **Check.** With a correct query, you should see 1395 rows and 1395 distinct tickets: 1,391 `resolved`, 3 `escalated` and 1 `reopened`.

Mei saves the checks next to the query, as `queries/36-latest-outcome-check.sql`. Run it: all three numbers must be equal.

In [ ]:
sql("""
-- Checks for 35-latest-outcome.sql: all three numbers must be equal.
WITH unique_tickets AS (
    SELECT DISTINCT ticket_id
    FROM tickets
),
ranked AS (
    SELECT
        o.ticket_id,
        o.outcome_id,
        o.recorded_at,
        o.status,
        ROW_NUMBER() OVER (
            PARTITION BY o.ticket_id
            ORDER BY o.recorded_at DESC, o.outcome_id DESC
        ) AS rn
    FROM outcomes AS o
    JOIN unique_tickets AS u
        ON u.ticket_id = o.ticket_id
),
latest AS (
    SELECT ticket_id, outcome_id, recorded_at, status
    FROM ranked
    WHERE rn = 1
)
SELECT
    (SELECT COUNT(*) FROM latest) AS latest_rows,
    (SELECT COUNT(DISTINCT ticket_id) FROM latest) AS distinct_tickets,
    (SELECT COUNT(*) FROM unique_tickets
     WHERE ticket_id IN (SELECT ticket_id FROM outcomes)) AS tickets_with_events;
""")

> **Check.** You should see `1395`, `1395`, `1395`. The join to `unique_tickets` leaves out the 4 events of missing tickets, `DISTINCT` stops the 14 repeated ticket rows from doubling events, and `outcome_id DESC` decides the 33 ties: the event logged last wins.

## 9. Reproduce the extraction

An extraction is reproducible when a second run on the same snapshot gives the same files. Run the script again, then run the next cell. It compares the new manifest with the one from section 6, and computes the SHA-256 of the two CSV files.

In [ ]:
!"{sys.executable}" scripts/extract.py

In [ ]:
again = json.loads(Path("data/extract/manifest.json").read_text(encoding="utf-8"))
changed = [key for key in again if again[key] != full[key]]
print("Parts of the manifest that changed:", changed)
for file, expected in [("data/extract/tickets.csv", "96786af9e2ebb2492219867ac302f57fabdefb4c53fb60a1071d9412e70e6f39"),
                       ("data/extract/outcomes.csv", "60c828f37db48a01ac6f003f5879a2c3ed5254f2c0e12816f814ed661c4067b4")]:
    sha = hashlib.sha256(Path(file).read_bytes()).hexdigest()
    expect(f"the SHA-256 of {file}", sha, expected)

> **Check.** You should see `['extracted_at']` (unless both runs were in the same second: then `[]`), and `Check passed` for both CSV files. The file fingerprints depend only on the snapshot, the selection and the query results, so they match the lesson's values on every computer.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/#module-check). They count toward your certificate when you are signed in and enrolled.

In this module you made your queries readable, correct over time, and repeatable. Named parameters keep values out of the SQL text: no broken quotes, no injection, and the same query fingerprint for every selection. `scripts/extract.py` writes the extraction and a manifest: the snapshot, the selection, the query SHA-256 values and the row counts. Run it twice, and the files are the same.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Profile before changing](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.